# Echo Chamber Co-Evolution Framework
## Pipeline Completa: Fase 0 → Fase 1 → Fase 2 → Fase 3

Questo notebook esegue la pipeline completa del framework in sequenza.

| Fase | Titolo | Moduli chiave |
|------|--------|---------------|
| **0** | Setup & Baseline | `data_loader`, `extractor`, `community`, `metrics` |
| **1** | Logica Agente | `agent`, `llm_client`, `state_machine`, `seeder` |
| **2** | Co-evoluzione | `orchestrator`, `gnn`, `rewirer`, `checkpoint` |
| **3** | CELF Fact-Checking | `celf`, `injector`, `influence/metrics` |

> ⚙️ **Configurazione**: modifica `config.yaml` o i parametri nella cella di setup per personalizzare la simulazione.

## 0. Setup Ambiente

In [1]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GEMINI_API_KEY")
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

os.environ["GEMINI_API_KEY"] = secret_value_0
os.environ["HF_TOKEN"] = secret_value_1

print("✅ Variabili d'ambiente impostate:")
print(f"   GEMINI_API_KEY = {'*' * 8}{secret_value_0[-4:]}")
print(f"   HF_TOKEN       = {'*' * 8}{secret_value_1[-4:]}")

✅ Variabili d'ambiente impostate:
   GEMINI_API_KEY = ********QdUk
   HF_TOKEN       = ********GVzS


In [2]:
import sys
import os
from pathlib import Path

# --- Path setup (compatibile locale + Kaggle) ---
if Path('/kaggle/working').exists():
    # Kaggle: clona il repo nella working dir
    PROJECT_ROOT = Path('/kaggle/working/progetto')
    if not PROJECT_ROOT.exists():
        os.system('git clone https://github.com/stefaano19/progettoASM /kaggle/working/progetto')
        os.system('pip install -r /kaggle/working/progetto/requirements.txt')
else:
    # Locale: la project root è due livelli sopra il notebook
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

Project root: /kaggle/working/progetto
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 0.1 Setup vLLM (LLM API Locale)
Installa vLLM, avvia l'API server in background distribuito sulle 2 T4 e scarica il modello Llama 3.

In [ ]:
!pip install vllm

import subprocess
cmd = (
    "nohup python -m vllm.entrypoints.openai.api_server "
    "--model unsloth/llama-3-8b-Instruct "
    "--tensor-parallel-size 2 "
    "--gpu-memory-utilization 0.75 "
    "--dtype half "
    "--max-model-len 2048 "
    "> vllm.log 2>&1 &"
)

subprocess.Popen(cmd, shell=True)


<Popen: returncode: None args: 'nohup python -m vllm.entrypoints.openai.api_...>

In [5]:
import requests, time

# vLLM impiega qualche secondo/minuto ad avviarsi: meglio fare retry
ok = False
for _ in range(120):  # Aumentato a 10 minuti per il download del modello
    try:
        response = requests.get("http://localhost:8000/v1/models", timeout=2)
        if response.status_code == 200:
            ok = True
            break
    except Exception:
        pass
    time.sleep(5)

if ok:
    print(f"✅ vLLM è online! Modelli: {[m['id'] for m in response.json().get('data', [])]}")
else:
    print("❌ Errore: vLLM non sta rispondendo (controlla vllm.log)")
    # utile per debug:
    !tail -n 50 vllm.log

✅ vLLM è online! Modelli: ['unsloth/llama-3-8b-Instruct']


In [ ]:
# Parametri globali della pipeline
# Modifica qui invece di editare config.yaml

CONFIG_PATH       = 'config.yaml'
USE_MOCK_LLM      = False       # False = usa il vero LLM
PHASE2_STEPS      = 20          # N. step da eseguire IN QUESTA SESSIONE (non totali!)
PHASE3_STEPS      = 5           # Step post-intervento
CELF_BUDGET_K     = 10          # Fact-checker da iniettare
SKIP_DOWNLOAD     = True        # True se ogbl-collab e' gia' in cache
RESUME_FROM_CKPT  = False       # True per riprendere da checkpoint Fase 2

# --- Checkpoint cross-versione Kaggle ---
# Se stai riprendendo da una versione precedente:
#   1. Aggiungi l'output della versione precedente come dataset input
#   2. Imposta PREV_VERSION_CKPT_INPUT al path del file .pkl
PREV_VERSION_CSV_INPUT  = None  # Path del metrics_history.csv dalla versione precedente
#      es. '/kaggle/input/nome-del-tuo-output/checkpoint_latest.pkl'
#   3. Imposta RESUME_FROM_CKPT = True
PREV_VERSION_CKPT_INPUT = None  # Path del .pkl dalla versione precedente (None = prima run)
PREV_VERSION_CSV_INPUT  = None  # Path del metrics_history.csv dalla versione precedente

print('Parametri pipeline:')
print(f'  CONFIG_PATH             = {CONFIG_PATH}')
print(f'  USE_MOCK_LLM            = {USE_MOCK_LLM}')
print(f'  PHASE2_STEPS            = {PHASE2_STEPS}')
print(f'  PHASE3_STEPS            = {PHASE3_STEPS}')
print(f'  CELF_BUDGET_K           = {CELF_BUDGET_K}')
print(f'  SKIP_DOWNLOAD           = {SKIP_DOWNLOAD}')
print(f'  RESUME_FROM_CKPT        = {RESUME_FROM_CKPT}')
print(f'  PREV_VERSION_CKPT_INPUT = {PREV_VERSION_CKPT_INPUT}')print(f'  PREV_VERSION_CSV_INPUT  = {PREV_VERSION_CSV_INPUT}')

In [ ]:
# Carica configurazione e setup globale
from src.utils.config import load_config
from src.utils.seed import set_all_seeds
from src.utils.logger import setup_logging
import logging

cfg = load_config(CONFIG_PATH)

# --- Override Locale --- 
cfg.llm.backend = "local"
cfg.llm.local.model = "unsloth/llama-3-8b-Instruct"
cfg.subgraph.target_nodes = 300000  
# -----------------------
setup_logging('INFO')
set_all_seeds(cfg.execution.random_seed)

print(f'Config caricata: hash={cfg.config_hash}')
print(f'Random seed: {cfg.execution.random_seed}')
print(f'Embedding dim: {cfg.gnn.embedding_dim}')
print(f'Max steps: {cfg.simulation.max_steps}')

[14:52:56] INFO     [Seed] Tutti i seed impostati a: 42

Config caricata: hash=15b5dc7edbd7
Random seed: 42
Embedding dim: 128
Max steps: 20


In [ ]:
from src.agents.llm_client import LLMClient
client = LLMClient.from_config(cfg)
resp = client.chat([{"role": "user", "content": "Rispondi solo con {\"ok\": true}"}])
print(resp.content, resp.is_fallback)

In [9]:
# --- SPECIFICHE DEL MODELLO (Llama 3 8B) ---
layers = 32
kv_heads = 8         # GQA (Grouped Query Attention) fa risparmiare moltissima RAM!
head_dim = 128
bytes_per_param = 2  # FP16 (16-bit) = 2 byte per parametro

# 1. Calcolo KV Cache per singolo token
# Formula: 2 (Key & Value) * bytes_per_param * layers * kv_heads * head_dim
bytes_per_token = 2 * bytes_per_param * layers * kv_heads * head_dim
kb_per_token = bytes_per_token / 1024

print(f"🔹 Dimensione KV Cache per singolo token: {kb_per_token:.1f} KB")

# --- PARAMETRI HARDWARE (2x T4 su Kaggle) ---
total_vram_gb = 32         # 2 x 16GB
model_weights_gb = 15      # I pesi dell'8B in FP16 occupano circa 15-16GB
cuda_overhead_gb = 1       # Riserviamo 1GB fisiologico per PyTorch/CUDA

vram_for_kv_gb = total_vram_gb - model_weights_gb - cuda_overhead_gb
print(f"🔹 VRAM libera allocata alla KV Cache: ~{vram_for_kv_gb} GB")

# 2. Quanti token possiamo tenere in memoria simultaneamente?
max_tokens_in_memory = (vram_for_kv_gb * (1024**3)) / bytes_per_token
print(f"🔹 Token massimi stipabili nella VRAM: {int(max_tokens_in_memory):,}")


# --- STIMA DEL BATCH SIZE ---
# Quanti token "pesa" in media la chiamata di un tuo Agente? 
# (Prompt in ingresso + JSON in uscita)
avg_tokens_per_request = 500  

max_batch_size = max_tokens_in_memory / avg_tokens_per_request

print("\n" + "-" * 55)
print(f"🚀 RICHIESTE IN PARALLELO GESTIBILI (BATCH SIZE): {int(max_batch_size)}")
print("-" * 55 + "\n")

valore_consigliato = int(max_batch_size) + 50

print("✅ Nel tuo config.yaml, imposta:")
print(f"   max_concurrent_requests: {valore_consigliato}")
print("\n💡 Info: Abbiamo aggiunto +50 per assicurarci che la coda di Python")
print("sia leggermente più grande della memoria della GPU, in modo da non avere MAI tempi morti.")

🔹 Dimensione KV Cache per singolo token: 128.0 KB
🔹 VRAM libera allocata alla KV Cache: ~16 GB
🔹 Token massimi stipabili nella VRAM: 131,072

-------------------------------------------------------
🚀 RICHIESTE IN PARALLELO GESTIBILI (BATCH SIZE): 262
-------------------------------------------------------

✅ Nel tuo config.yaml, imposta:
   max_concurrent_requests: 312

💡 Info: Abbiamo aggiunto +50 per assicurarci che la coda di Python
sia leggermente più grande della memoria della GPU, in modo da non avere MAI tempi morti.


---
## Fase 0 — Setup & Baseline

Carica `ogbl-collab`, estrae il sottografo, rileva community, calcola metriche baseline.

In [10]:
import numpy as np
import networkx as nx

# --- 1. Caricamento dati ---
from src.graph.data_loader import load_collab_graph, graph_summary

print('⏳ Caricamento ogbl-collab...')
G_full, node_features = load_collab_graph(cfg)
summary = graph_summary(G_full)
print(f'✅ Grafo completo: {summary["num_nodes"]:,} nodi | {summary["num_edges"]:,} archi')

⏳ Caricamento ogbl-collab...


[14:52:57] INFO     [DataLoader] Caricamento grafo da cache:                                                       
                    /kaggle/working/progetto/data/processed/full_graph.gpickle

✅ Grafo completo: 235,868 nodi | 967,632 archi


In [11]:
# --- 2. Estrazione sottografo ---
from src.graph.extractor import extract_subgraph

print(f'⏳ Estrazione sottografo (target={cfg.subgraph.target_nodes} nodi)...')
subG, sub_features, node_map = extract_subgraph(G_full, node_features, cfg)
print(f'✅ Sottografo: {subG.number_of_nodes()} nodi | {subG.number_of_edges()} archi')
print(f'   Densità: {nx.density(subG):.4f}')

⏳ Estrazione sottografo (target=300000 nodi)...


[14:52:59] INFO     [Extractor] Caricamento sottografo da cache:                                                   
                    /kaggle/working/progetto/data/processed/subgraph.gpickle

✅ Sottografo: 232865 nodi | 961883 archi
   Densità: 0.0000


In [12]:
# --- 3. Community detection ---
from src.graph.community import detect_communities, community_stats

print(f'⏳ Community detection (algoritmo={cfg.community.algorithm})...')
community_map, n_comm, q = detect_communities(subG, cfg)
stats = community_stats(subG, community_map)
print(f'✅ Community: {n_comm} trovate | Q={q:.4f}')
print(f'   Dimensioni: min={stats["min_size"]} max={stats["max_size"]} media={stats["mean_size"]:.1f}')

⏳ Community detection (algoritmo=louvain)...


[14:53:01] INFO     [Community] Community detection con algoritmo='louvain'...

KeyboardInterrupt: 

In [ ]:
# --- 4. Metriche baseline ---
from src.graph.metrics import compute_all_metrics, compute_centralities

print('⏳ Calcolo metriche baseline...')
baseline_metrics = compute_all_metrics(subG, cfg, community_map=community_map)
centralities = compute_centralities(subG, cfg)

print('\n📊 METRICHE BASELINE (Fase 0)')
print('=' * 40)
for k, v in baseline_metrics.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:.4f}')
    elif isinstance(v, int):
        print(f'  {k:30s}: {v:,}')

# Top-5 PageRank
top_pr = sorted(
    [(n, d.get('pagerank', 0)) for n, d in centralities.items()],
    key=lambda x: x[1], reverse=True
)[:5]
print(f'\n  Top-5 PageRank: {[(n, f"{pr:.4f}") for n, pr in top_pr]}')

---
## Fase 1 — Logica Agente

Inizializza agenti LLM, seleziona pazienti zero, esegue un singolo step di test.

In [ ]:
from src.agents.llm_client import MockLLMClient
from src.agents.state_machine import StateMachine
from src.agents.seeder import Seeder
from src.agents.agent import Agent
from src.graph.network_manager import NetworkManager
from src.gnn.embeddings import EmbeddingManager

# --- Embeddings ---
em = EmbeddingManager(cfg)
embedding_path = cfg.project_root / cfg.gnn.embedding_file
if embedding_path.exists():
    embeddings = em.load()
else:
    embeddings = em.initialize(subG, sub_features)
    em.save(embeddings)
print(f'✅ Embeddings: shape={embeddings.shape}')

# --- NetworkManager ---
nm = NetworkManager(subG, cfg, community_map=community_map, node_features=embeddings)
print(f'✅ NetworkManager: {nm.num_nodes} nodi | {nm.num_edges} archi')

In [ ]:
# --- Selezione pazienti zero ---
seeder = Seeder(cfg, strategy=cfg.simulation.seeder_strategy)
patient_zero_ids = seeder.select(subG, centralities, community_map)
seeder.inject(nm, patient_zero_ids, initial_state='I')

print(f'✅ Pazienti zero selezionati: {len(patient_zero_ids)} nodi')
print(f'   IDs: {patient_zero_ids}')

# Report dettagliato
seed_report = seeder.describe_seeds(patient_zero_ids, centralities, community_map)
for s in seed_report:
    print(f"   Node {s['node_id']:4d} | comm={s['community']} | PR={s['pagerank']:.5f} | deg={s['degree']}")

In [ ]:
# --- Inizializzazione agenti ---
llm_client = MockLLMClient(seed=cfg.execution.random_seed)
state_machine = StateMachine.from_config(cfg)

agents = {}
for node_id in nm.nodes:
    initial_state = nm.get_state(node_id)
    agent = Agent(
        node_id=node_id,
        cfg=cfg,
        llm_client=llm_client,
        state_machine=state_machine,
        initial_state=initial_state,
    )
    centrality_val = centralities.get(node_id, {}).get('degree_centrality', 0.0)
    agent.initialize(
        community=community_map.get(node_id, 0),
        centrality=centrality_val,
        network_manager=nm,
    )
    agents[node_id] = agent

print(f'✅ {len(agents)} agenti inizializzati')

# Test: un singolo step su un agente infetto
pz = patient_zero_ids[0]
decision = agents[pz].step(0, nm)
print(f'\n🧪 Test agente {pz}: {decision.old_state} → {decision.new_state} | susc={decision.susceptibility:.3f}')

---
## Fase 2 — Co-evoluzione (Agenti ↔ GNN ↔ Rewiring)

Loop temporale completo: agenti → GNN fine-tuning → rewiring topologico.

In [ ]:
# ============================================================
# IMPORT CHECKPOINT DA VERSIONE PRECEDENTE (cross-session Kaggle)
# ============================================================
# Se PREV_VERSION_CKPT_INPUT e' impostato, copia il checkpoint
# dall'output della versione precedente nella directory checkpoints
# del progetto, cosi' RESUME_FROM_CKPT=True trovera' il file.
import shutil
from pathlib import Path

if RESUME_FROM_CKPT and PREV_VERSION_CKPT_INPUT is not None:
    src_ckpt = Path(PREV_VERSION_CKPT_INPUT)
    if not src_ckpt.exists():
        raise FileNotFoundError(f'Checkpoint non trovato: {src_ckpt}')
    ckpt_dir = PROJECT_ROOT / 'results' / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    dst = ckpt_dir / src_ckpt.name
    shutil.copy2(src_ckpt, dst)
    print(f'\u2705 Checkpoint importato: {src_ckpt.name} \u2192 {dst}')
elif RESUME_FROM_CKPT:
    # Controlla se esiste gia' un checkpoint locale
    ckpt_dir = PROJECT_ROOT / 'results' / 'checkpoints'
    existing = sorted(ckpt_dir.glob('ckpt_step_*.pkl')) if ckpt_dir.exists() else []
    if existing:
        print(f'\u2705 Checkpoint locale trovato: {existing[-1].name}')
    else:
        print('\u26a0\ufe0f  RESUME_FROM_CKPT=True ma nessun checkpoint trovato.')
        print('   Imposta PREV_VERSION_CKPT_INPUT per importare da versione precedente.')
        RESUME_FROM_CKPT = False  # fallback: nuova run
else:
    print('\u2139\ufe0f  Prima sessione, parto da zero.')

In [ ]:
# ============================================================
# IMPORT CSV METRICHE DA VERSIONE PRECEDENTE
# ============================================================
# Se PREV_VERSION_CSV_INPUT e' impostato, copia il CSV dalla
# versione precedente nella directory results/ del progetto.
# L'orchestratore aprira' il CSV in append mode e accodera' i nuovi step,
# mantenendo la storia completa di tutte le sessioni.
import shutil
from pathlib import Path

results_dir = PROJECT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
csv_dst = results_dir / 'metrics_history.csv'

if RESUME_FROM_CKPT and PREV_VERSION_CSV_INPUT is not None:
    src_csv = Path(PREV_VERSION_CSV_INPUT)
    if src_csv.exists():
        shutil.copy2(src_csv, csv_dst)
        import pandas as pd
        df_prev = pd.read_csv(csv_dst)
        print(f'\u2705 CSV metriche importato: {src_csv.name}')
        print(f'   Righe precedenti: {len(df_prev)} step (step {df_prev["step"].min()}..{df_prev["step"].max()})')
    else:
        print(f'\u26a0\ufe0f  CSV non trovato: {src_csv} \u2014 si parte da zero.')
elif RESUME_FROM_CKPT and csv_dst.exists():
    import pandas as pd
    df_prev = pd.read_csv(csv_dst)
    print(f'\u2705 CSV locale trovato: {len(df_prev)} step precedenti')
else:
    print('\u2139\ufe0f  Nessun CSV precedente da importare (prima sessione).')

In [ ]:
import uuid
from src.orchestrator import SimulationOrchestrator

print('⏳ Costruzione SimulationOrchestrator...')
orch = SimulationOrchestrator.build_from_config(
    cfg,
    use_mock_llm=USE_MOCK_LLM,
    resume=RESUME_FROM_CKPT,
)
print(f'✅ Orchestratore pronto')
print(f'   Nodi: {orch.network_manager.num_nodes}')
print(f'   Archi: {orch.network_manager.num_edges}')

In [ ]:
# Tracking metriche per step (per i grafici finali)
metrics_history = []

# --- Resume: calcola il range corretto ---
# orch.resume_step = 0 se nuova run, = ckpt_step+1 se resume
start_step = orch.resume_step
end_step   = start_step + PHASE2_STEPS

print(f'\U0001f680 Avvio loop co-evolutivo: step {start_step} \u2192 {end_step - 1} ({PHASE2_STEPS} step questa sessione)...')
print(f'   (PHASE2_STEPS conta gli step di QUESTA sessione, non il totale)')
print('=' * 65)
print(f'{"Step":>5}  {"S":>6}  {"I":>6}  {"R":>6}  {"F":>6}  {"ECI":>8}  {"Loss":>8}  {"Edges":>7}')
print('-' * 65)

for t in range(start_step, end_step):
    step_metrics = orch._run_step(t)
    s_counts = orch.state_summary
    metrics_history.append({'step': t, **step_metrics, **s_counts})

    print(
        f'{t:>5}  {s_counts.get("S",0):>6}  {s_counts.get("I",0):>6}  '
        f'{s_counts.get("R",0):>6}  {s_counts.get("F",0):>6}  '
        f'{step_metrics.get("echo_chamber_index") or 0.0:>8.4f}  '
        f'{step_metrics.get("gnn_loss") or 0.0:>8.4f}  '
        f'{step_metrics.get("num_edges") or 0:>7}'
    )

print('=' * 65)
print(f'\u2705 Loop completato. Step eseguiti: {start_step}\u2013{end_step - 1}')
print(f'   Ultimo checkpoint automatico: ckpt_step_{orch.current_step:04d}.pkl')

In [ ]:
# ============================================================
# EXPORT CHECKPOINT PER LA PROSSIMA VERSIONE KAGGLE
# ============================================================
# Kaggle conserva i file in /kaggle/working/ come output della versione.
# Copiamo il checkpoint piu' recente con nome fisso 'checkpoint_latest.pkl'
# cosi' nella prossima sessione sara' facile referenziarlo.
import shutil
import pandas as pd
from pathlib import Path

ckpt_dir = PROJECT_ROOT / 'results' / 'checkpoints'
ckpts = sorted(ckpt_dir.glob('ckpt_step_*.pkl'))

if ckpts:
    latest = ckpts[-1]

    # Copia con nome fisso per facile recupero nella versione successiva
    export_ckpt = Path('/kaggle/working/checkpoint_latest.pkl')
    shutil.copy2(latest, export_ckpt)

    # Esporta anche le metriche cumulative come CSV
    export_csv = Path('/kaggle/working/metrics_history.csv')
    pd.DataFrame(metrics_history).to_csv(export_csv, index=False)

    print(f'\u2705 Checkpoint esportato : {export_ckpt}')
    print(f'   Step salvato         : {orch.current_step}')
    print(f'\u2705 Metriche esportate   : {export_csv}')
    print()
    print('\U0001f4cc PROSSIMA SESSIONE \u2014 imposta nel notebook:')
    print(f'   RESUME_FROM_CKPT        = True')
    print(f'   PREV_VERSION_CKPT_INPUT = "/kaggle/input/<nome-output>/checkpoint_latest.pkl"')
    print(f'   PHASE2_STEPS            = <quanti step vuoi fare>')
else:
    print('\u26a0\ufe0f  Nessun checkpoint trovato da esportare.')

In [ ]:
# Chiudi il file CSV delle metriche in modo pulito
orch.close()
print('\u2705 Orchestratore chiuso (CSV metriche flushed).')

In [ ]:
# Visualizzazione evoluzione nel tempo
try:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    steps = [m['step'] for m in metrics_history]
    n_S = [m.get('S', 0) for m in metrics_history]
    n_I = [m.get('I', 0) for m in metrics_history]
    n_R = [m.get('R', 0) for m in metrics_history]
    n_F = [m.get('F', 0) for m in metrics_history]
    eci = [m.get('echo_chamber_index') or 0.0 for m in metrics_history]
    loss= [m.get('gnn_loss') or 0.0 for m in metrics_history]
    edges=[m.get('num_edges') or 0 for m in metrics_history]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Echo Chamber Co-Evolution — Fase 2', fontsize=13, fontweight='bold')
    
    # Plot 1: Dinamica stati
    axes[0].stackplot(steps, n_S, n_I, n_R, n_F,
                      labels=['S (Susceptible)', 'I (Infected)', 'R (Resistant)', 'F (Fact-Checker)'],
                      colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], alpha=0.8)
    axes[0].set_title('Dinamica degli Stati')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('N. Nodi')
    axes[0].legend(loc='upper left', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Echo Chamber Index
    axes[1].plot(steps, eci, color='#8e44ad', linewidth=2, marker='o', markersize=4)
    axes[1].fill_between(steps, eci, alpha=0.2, color='#8e44ad')
    axes[1].set_title('Echo Chamber Index (ECI)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('ECI')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 1)
    
    # Plot 3: Archi nel tempo (rewiring)
    axes[2].plot(steps, edges, color='#16a085', linewidth=2, marker='s', markersize=4)
    axes[2].fill_between(steps, edges, alpha=0.2, color='#16a085')
    axes[2].set_title('Evoluzione Archi (Rewiring)')
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('N. Archi')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    fig_path = cfg.project_root / cfg.paths.figures / 'phase2_evolution.png'
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'📊 Grafico salvato: {fig_path}')
    plt.show()

except ImportError:
    print('matplotlib non disponibile — skip grafici')

---
## Fase 3 — CELF Fact-Checking

Selezione ottimale dei seed fact-checker, iniezione e misurazione dell'impatto.

In [ ]:
from src.graph.metrics import compute_all_metrics
from src.agents.state_machine import StateMachine

# Snapshot pre-intervento
nm_ph3 = orch.network_manager
community_map_ph3 = nm_ph3._community_map

pre_belief_map = nm_ph3.get_belief_map()
pre_metrics = compute_all_metrics(nm_ph3.G, cfg, community_map_ph3, pre_belief_map)
pre_states = nm_ph3.get_all_states()
pre_counts = StateMachine.count_states(pre_states)
n_total = max(nm_ph3.num_nodes, 1)
pre_metrics['infection_rate'] = pre_counts['I'] / n_total
pre_metrics['n_S'] = pre_counts['S']
pre_metrics['n_I'] = pre_counts['I']
pre_metrics['n_R'] = pre_counts['R']
pre_metrics['n_F'] = pre_counts['F']

print('📊 BASELINE PRE-INTERVENTO')
print(f'  S={pre_counts["S"]}  I={pre_counts["I"]}  R={pre_counts["R"]}  F={pre_counts["F"]}')
print(f'  Infection Rate : {pre_metrics["infection_rate"]:.4f}')
print(f'  ECI            : {pre_metrics.get("echo_chamber_index") or 0.0:.4f}')
print(f'  Modularity Q   : {pre_metrics.get("modularity_q") or 0.0:.4f}')

In [ ]:
# --- CELF: selezione seed ---
from src.influence.celf import CELF

print(f'⏳ CELF: selezione {CELF_BUDGET_K} seed fact-checker...')
celf = CELF(cfg)
celf_seeds = celf.select(
    G=nm_ph3.G,
    budget_k=CELF_BUDGET_K,
    agent_states=nm_ph3.get_all_states(),
)
print(f'✅ CELF seeds: {celf_seeds}')

# Stima spread
estimated_spread = celf.estimate_spread(nm_ph3.G, celf_seeds, nm_ph3.get_all_states())
print(f'   Spread stimato: {estimated_spread:.1f} nodi ({estimated_spread/n_total*100:.1f}%)')

In [ ]:
# --- Iniezione fact-checker ---
from src.influence.injector import FactCheckerInjector

injector = FactCheckerInjector(cfg)
injected = injector.inject(nm_ph3, celf_seeds, step=orch.current_step)
print(f'✅ Fact-checker iniettati: {len(injected)} nodi → {injected}')

# Aggiorna stato agenti in memoria
from src.agents.agent import AgentState
for node_id in injected:
    if node_id in orch._agents:
        orch._agents[node_id]._state = AgentState.from_str('F')

In [ ]:
# --- Step post-intervento ---
print(f'⏳ Esecuzione {PHASE3_STEPS} step post-intervento...')
post_history = []

start_step = orch.current_step + 1
for t in range(start_step, start_step + PHASE3_STEPS):
    step_metrics = orch._run_step(t)
    s_counts = orch.state_summary
    post_history.append({'step': t, **step_metrics, **s_counts})
    print(
        f'  Step {t}: S={s_counts.get("S",0)} I={s_counts.get("I",0)} '
        f'R={s_counts.get("R",0)} F={s_counts.get("F",0)} '
        f'ECI={step_metrics.get("echo_chamber_index") or 0.0:.4f}'
    )

print('✅ Step post-intervento completati')

In [ ]:
# --- Report completo ---
from src.influence.metrics import compute_full_influence_report

final_report = compute_full_influence_report(
    G=nm_ph3.G,
    agent_states=nm_ph3.get_all_states(),
    community_map=community_map_ph3,
    baseline_metrics=pre_metrics,
    cfg=cfg,
)

print('\n' + '=' * 65)
print('CONFRONTO BEFORE / AFTER INTERVENTO CELF')
print('=' * 65)
print(f'{"Metrica":<25}  {"PRIMA":>10}  {"DOPO":>10}  {"DELTA":>10}')
print('-' * 65)

metrics_to_compare = [
    ('Infection Rate',      'infection_rate',       '%.4f'),
    ('Echo Chamber Idx',    'echo_chamber_index',   '%.4f'),
    ('Modularity Q',        'modularity_q',         '%.4f'),
    ('Belief Polarisation', 'belief_polarisation',  '%.4f'),
    ('Nodi S',              'n_S',                  '%d'),
    ('Nodi I',              'n_I',                  '%d'),
    ('Nodi R',              'n_R',                  '%d'),
    ('Nodi F',              'n_F',                  '%d'),
]

for label, key, fmt in metrics_to_compare:
    before = pre_metrics.get(key)
    after  = final_report.get(key)
    delta  = final_report.get(f'delta_{key}')
    before_s = (fmt % before) if before is not None else 'n/a'
    after_s  = (fmt % after)  if after  is not None else 'n/a'
    delta_s  = (f'{delta:+.4f}' if isinstance(delta, float) else 'n/a')
    print(f'{label:<25}  {before_s:>10}  {after_s:>10}  {delta_s:>10}')

print('-' * 65)
print(f'{"Fact-Checker Spread":<25}  {"":>10}  {final_report.get("fcs", 0.0):>10.4f}')
print(f'{"Avg Reach per FC":<25}  {"":>10}  {final_report.get("avg_reach_per_fc", 0.0):>10.1f}')
print('=' * 65)

In [ ]:
# --- Visualizzazione Phase 3 ---
try:
    import matplotlib.pyplot as plt
    
    # Combina metriche storia completa
    all_history = metrics_history + post_history
    all_steps = [m['step'] for m in all_history]
    all_n_I = [m.get('I', m.get('n_I', 0)) for m in all_history]
    all_n_F = [m.get('F', m.get('n_F', 0)) for m in all_history]
    all_eci = [m.get('echo_chamber_index') or 0.0 for m in all_history]
    
    injection_step = start_step - 1
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Echo Chamber Framework — Effetto Intervento CELF', fontsize=13, fontweight='bold')
    
    # Plot: Nodi I e F nel tempo
    ax1.plot(all_steps, all_n_I, color='#e74c3c', linewidth=2, label='I (Infected)', marker='o', ms=4)
    ax1.plot(all_steps, all_n_F, color='#f39c12', linewidth=2, label='F (Fact-Checker)', marker='s', ms=4)
    ax1.axvline(x=injection_step, color='black', linestyle='--', alpha=0.7, label=f'Iniezione FC (step {injection_step})')
    ax1.set_title('Diffusione: Infetti vs Fact-Checker')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('N. Nodi')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Plot: ECI nel tempo
    ax2.plot(all_steps, all_eci, color='#8e44ad', linewidth=2, marker='o', ms=4)
    ax2.axvline(x=injection_step, color='black', linestyle='--', alpha=0.7, label=f'Iniezione FC')
    ax2.fill_between(all_steps, all_eci, alpha=0.15, color='#8e44ad')
    ax2.set_title('Echo Chamber Index nel Tempo')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('ECI')
    ax2.set_ylim(0, 1)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    fig_path = cfg.project_root / cfg.paths.figures / 'phase3_intervention.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'📊 Grafico salvato: {fig_path}')
    plt.show()

except ImportError:
    print('matplotlib non disponibile — skip grafici')

---
## Riepilogo Pipeline

In [ ]:
import json

pipeline_summary = {
    'config_hash': cfg.config_hash,
    'seed': cfg.execution.random_seed,
    'phase0': {
        'n_nodes': subG.number_of_nodes(),
        'n_edges': subG.number_of_edges(),
        'n_communities': n_comm,
        'modularity_q_baseline': round(q, 4),
        'echo_chamber_index_baseline': round(baseline_metrics.get('echo_chamber_index') or 0.0, 4),
    },
    'phase1': {
        'n_patient_zeros': len(patient_zero_ids),
        'patient_zero_ids': patient_zero_ids,
        'strategy': cfg.simulation.seeder_strategy,
    },
    'phase2': {
        'n_steps': PHASE2_STEPS,
        'llm_mode': 'mock' if USE_MOCK_LLM else 'api',
        'final_state_counts': pre_counts,
        'infection_rate_post_phase2': round(pre_metrics['infection_rate'], 4),
        'eci_post_phase2': round(pre_metrics.get('echo_chamber_index') or 0.0, 4),
    },
    'phase3': {
        'celf_seeds': celf_seeds,
        'injected_nodes': injected,
        'budget_k': CELF_BUDGET_K,
        'n_post_steps': PHASE3_STEPS,
        'fcs': round(final_report.get('fcs', 0.0), 4),
        'delta_infection_rate': round(final_report.get('delta_infection_rate', 0.0), 4),
        'delta_eci': round(final_report.get('delta_echo_chamber_index', 0.0), 4),
        'final_n_F': final_report.get('n_F', 0),
    },
}

summary_path = cfg.project_root / cfg.paths.results / 'pipeline_summary.json'
with open(summary_path, 'w') as f:
    json.dump(pipeline_summary, f, indent=2)

print('\n' + '=' * 60)
print('PIPELINE COMPLETATA ✅')
print('=' * 60)
print(f'  Nodi simulati    : {subG.number_of_nodes()}')
print(f'  Archi iniziali   : {subG.number_of_edges()}')
print(f'  Community        : {n_comm} (Q baseline={q:.4f})')
print(f'  Step Fase 2      : {PHASE2_STEPS}')
print(f'  Infection rate   : {pre_metrics["infection_rate"]:.4f} → {final_report.get("infection_rate", 0.0):.4f}')
print(f'  ECI              : {pre_metrics.get("echo_chamber_index") or 0.0:.4f} → {final_report.get("echo_chamber_index") or 0.0:.4f}')
print(f'  CELF seeds       : {len(celf_seeds)} nodi')
print(f'  FCS post-int.    : {final_report.get("fcs", 0.0):.4f}')
print('=' * 60)
print(f'Riepilogo salvato: {summary_path}')